In [ ]:
from simsopt.mhd import Boozer,Vmec
from simsopt.geo import Surface,SurfaceRZFourier

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import neo
from neo import NeoContext,neo_surfaces_from_simsopt_boozer
import time
import numpy as np
from pathlib import Path


mgrid_candidates = [
    Path("tests/test_file/mgrid_c09r00.nc"),
    Path("test_file/mgrid_c09r00.nc"),
]
mgrid_path = next((p for p in mgrid_candidates if p.exists()), None)
if mgrid_path is None:
    raise FileNotFoundError(
        "Cannot find mgrid file. Tried: " + ", ".join(str(p) for p in mgrid_candidates)
    )


# initial_rz = (1.57,0)
# mgrid_filename = str(mgrid_path)
# extcur = None
# vmec_path = "/home/zkg/ripplepy/tests/test_file/wout_ncsx_c09r00_free.nc"
initial_rz = (1.26,0)
mgrid_filename = '/home/zkg/CN_H1_scan_fieldlines/H1_design/coils/mgrid_h1_design.nc'
extcur = [50000, 5000, 1, -80000, -40000]
vmec_path = "/home/zkg/ripplepy/tests/test_file/wout_h1_design.nc"

sur_idx = np.linspace(0.1, 0.5, 10)

RZ_points = []  # 每个元素是 [R, phi, Z]
for s in sur_idx:
    surface = SurfaceRZFourier.from_wout(vmec_path, s)
    RphiZ = surface.cross_section(phi=0)[0]
    RZ = RphiZ[[0, 2]]   # shape: (Npts, 3)
    RZ_points.append(RZ)             # 第一个点: (3,)

RZ_points = np.asarray(RZ_points)       # shape: (len(sur_idx), 3) [R1,Z1], ...]

vmec = Vmec(str(vmec_path))
boozer = Boozer(vmec)
boozer.mpol = 128
boozer.ntor = 128
# ns_list =  np.array([2, 3, 4, 5, 6, 7, 8, 9, 10])
# boozer.register(ns_list/100)
boozer.register(np.linspace(0.1, 1.0, 10))
boozer.bx.verbose =True
boozer.run()

boozer.save(str(mgrid_path))

neoclass = neo.from_simsopt_boozer(boozer)
ctx = NeoContext()
ctx.set_boozer(neoclass)
surfaces = neo_surfaces_from_simsopt_boozer(boozer)
print('Surfaces from simsopt Boozer:', surfaces)
ctx.set_flux_surfaces(surfaces.tolist())
ctx.set_resolution(theta_n=200, phi_n=200)
ctx.set_mode_limits(max_m_mode=0, max_n_mode=0)
ctx.set_transport_options(
    npart=100,
    multra=1,
    acc_req=0.01,
    no_bins=100,
    nstep_per=50,
    nstep_min=500,
    nstep_max=5000,
    calc_nstep_max=0,
)
ctx.set_switches(ref_swi=2, eout_swi=2, calc_cur=0)
ctx.set_output_options(
    write_progress=0,
    write_output_files=0,
    write_integrate=0,
    write_diagnostic=0,
    suppress_file_io=True,
)

ctx.setup_grids()
ctx.run_all()

got_surfaces = ctx.surface_map()
got_epstot = ctx.epstot_profile()